# Generar Estructura

Thin notebook for route 1: Generar estructura.


## Load Package


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import zipfile
import importlib

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openpyxl", "ipywidgets"])

PACKAGE_ZIP = "lab_pipeline_package.zip"
PACKAGE_ZIP_PREFIX = "lab_pipeline_package"
NOTEBOOK_NAMES = [
    'Lab_App_Colab_UI.ipynb',
    "Lab_Main_Pipeline.ipynb",
    "Generar_Estructura.ipynb",
    "Lab_Group_Manager.ipynb",
    "Import_Teammates_to_Notas.ipynb",
]


def running_in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False


def mount_drive_if_colab():
    if not running_in_colab():
        return
    from google.colab import drive
    drive.mount("/content/drive")


def find_notebook_workspace():
    if not running_in_colab():
        return Path.cwd()

    mydrive = Path("/content/drive/MyDrive")
    matches = []
    if mydrive.exists():
        for notebook_name in NOTEBOOK_NAMES:
            matches.extend(mydrive.rglob(notebook_name))

    if matches:
        newest = max(matches, key=lambda path: path.stat().st_mtime)
        return newest.parent

    fallback_name = input(
        "No pude detectar la carpeta del notebook en Drive. "
        "Nombre de carpeta base [Generador_GNT]: "
    ).strip() or "Generador_GNT"
    workspace = mydrive / fallback_name
    workspace.mkdir(parents=True, exist_ok=True)
    return workspace


def force_workspace_first(workspace_dir):
    workspace_text = str(workspace_dir)
    sys.path[:] = [path for path in sys.path if path != workspace_text]
    sys.path.insert(0, workspace_text)


def clear_lab_pipeline_modules():
    for module_name in list(sys.modules):
        if module_name == "lab_pipeline" or module_name.startswith("lab_pipeline."):
            del sys.modules[module_name]


def add_lab_pipeline_to_path(workspace_dir):
    force_workspace_first(workspace_dir)
    if (workspace_dir / "lab_pipeline").exists():
        return workspace_dir
    for folder in [Path.cwd(), Path("/content")]:
        if (folder / "lab_pipeline").exists():
            if str(folder) not in sys.path:
                sys.path.append(str(folder))
            return folder
    return None


def find_uploaded_package(uploaded):
    for uploaded_name in uploaded:
        path = Path(uploaded_name)
        if path.suffix.lower() == ".zip" and path.stem.startswith(PACKAGE_ZIP_PREFIX):
            return path
    raise FileNotFoundError(
        "Debes subir el zip del paquete, por ejemplo lab_pipeline_package.zip. "
        "Si Colab lo renombra como lab_pipeline_package (1).zip tambien sirve."
    )


def upload_and_extract_package(workspace_dir):
    from google.colab import files

    print(f"Sube ahora {PACKAGE_ZIP}.")
    uploaded = files.upload()
    package_path = find_uploaded_package(uploaded)

    with zipfile.ZipFile(package_path, "r") as z:
        z.extractall(workspace_dir)

    print(f"Paquete instalado/actualizado en: {workspace_dir}")


mount_drive_if_colab()
WORKSPACE_DIR = find_notebook_workspace()
os.environ["LAB_PIPELINE_WORKSPACE_DIR"] = str(WORKSPACE_DIR)
print(f"Carpeta base de trabajo: {WORKSPACE_DIR}")

package_folder = add_lab_pipeline_to_path(WORKSPACE_DIR)

if package_folder:
    update = input(
        "lab_pipeline ya existe. Hay una nueva version del zip para actualizar? [s/N]: "
    ).strip().lower()
    if update in ("s", "si", "sí", "y", "yes"):
        upload_and_extract_package(WORKSPACE_DIR)
        package_folder = add_lab_pipeline_to_path(WORKSPACE_DIR)
else:
    if running_in_colab():
        print("Primera ejecucion: no encontre lab_pipeline en esta carpeta.")
        upload_and_extract_package(WORKSPACE_DIR)
        package_folder = add_lab_pipeline_to_path(WORKSPACE_DIR)
    else:
        raise ModuleNotFoundError("No encontre lab_pipeline en la carpeta actual.")

if not package_folder:
    raise ModuleNotFoundError("No pude cargar lab_pipeline despues de instalar el paquete.")

clear_lab_pipeline_modules()
force_workspace_first(WORKSPACE_DIR)
import lab_pipeline

print("lab_pipeline loaded.")
print(f"lab_pipeline file: {Path(lab_pipeline.__file__).resolve()}")


## Run Route 1


In [ ]:
from lab_pipeline.structure import run_structure_workflow

result = run_structure_workflow()
